# Preface

This module contains code for downloading three different datasets:
- MIMIC-IV
- MIMIC-CXR: completer and partial
- MIMIC-CXR-JPEG


### Initialize

In [2]:
%load_ext autoreload
%autoreload 2
from config.settings_data import DataSetsSettings
from utils import templates as T

profile = "default"

settings = DataSetsSettings.build(profile)

files_to_fix = []

# 1 `MIMIC-IV`

### 1.1 `Download Data`

In [7]:
from getpass import getpass
from utils.mimic_utils import download_mimic_iv

username = input("Enter PhysioNet username: ")
password = getpass("Enter PhysioNet password: ")

download_mimic_iv(
    username=username,
    password=password,
    target_dir=settings.MIMIC_IV_root,
    templates=T
)


--2026-07-23 11:48:24--  https://physionet.org/files/mimiciv/3.1/
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized

Username/Password Authentication Failed.

❌ wget exited with code {error_code}.


### 1.2 `Check files`

In [2]:
from utils.mimic_utils import checksum_validation_mimic_iv

validation_result = checksum_validation_mimic_iv(
    mimic_iv_path=settings.MIMIC_IV_root,
    templates=T
)


for error_type in validation_result:
    print(T.MESSAGE_MISSING_TYPE.format(
        error_type = error_type,
        error_num = len(validation_result[error_type])
        ))


files_to_fix = validation_result["wrong checksum"]


Number of correct checksum: 33
Number of wrong checksum: 0
Number of missing files: 0


### 1.3. `UnZip Files`

In [ ]:
from utils.mimic_utils import unpack_mimic_iv

unpack_mimic_iv(
    correct_files=validation_result['correct checksum']
)

print(T.MESSAGE_SUCCESS_UNZIP)

'\n# --- 0) Base folder ---\nif \'location\' not in locals():\n    location = Path(input("Enter download location (full path): ").strip()).expanduser().resolve()\n\nprint(f"\n📁 Using folder: {location}")\n\nchecksum_file = location / "SHA256SUMS.txt"\nif not checksum_file.exists():\n    raise FileNotFoundError(f"Checksum file not found: {checksum_file}")\n\n# --- 1) Read relative paths from SHA256SUMS.txt ---\nrel_paths = []\nwith open(checksum_file, "r", encoding="utf-8") as f:\n    for line in f:\n        line = line.strip()\n        if not line:\n            continue\n        # Robust split: first token is hash, the rest is the path (may contain spaces)\n        parts = line.split(maxsplit=1)\n        if len(parts) != 2:\n            continue\n        _, rel_path = parts\n        rel_paths.append(rel_path.strip())\n\n# --- 2) Filter only .gz files ---\ngz_paths = [rp for rp in rel_paths if rp.endswith(".gz")]\nprint(f"Found {len(gz_paths)} .gz files to process")\n\n# --- 3) Decompre

# 2 `Complete MIMIC-CXR Download`

This particular code will download the entire dataset. It can be very time consuming.

To download a portion of the dataset go to Section 3 `Partial MIMIC-CXR Download`

### 2.1 `Download Data`

In [ ]:
from getpass import getpass
from utils.mimic_utils import download_mimic_cxr

username = input("Enter PhysioNet username: ")
password = getpass("Enter PhysioNet password: ")

download_mimic_cxr(
    username=username,
    password=password,
    target_dir=settings.MIMIC_CXR_root,
    templates=T
)


### 2.2 `Check files`

In [8]:
from utils.mimic_utils import checksum_validation_mimic_iv

validation_result = checksum_validation_mimic_iv(
    mimic_iv_path=settings.MIMIC_CXR_root,
    templates=T
)


for error_type in validation_result:
    print(T.MESSAGE_MISSING_TYPE.format(
        error_type = error_type,
        error_num = len(validation_result[error_type])
        ))

files_to_fix = validation_result["wrong checksum"]


Number of correct checksum: 87530
Number of wrong checksum: 0
Number of missing files: 517420


# 3 `Partial MIMIC-CXR Download`

### 3.1 `Download CHECKSUM`

In [ ]:
from getpass import getpass
from utils.mimic_utils import download_mimic_cxr_checksum

username = input("Enter PhysioNet username: ")
password = getpass("Enter PhysioNet password: ")

checksum_file = download_mimic_cxr_checksum(
    username=username,
    password=password,
    target_dir=settings.MIMIC_CXR_root,
    templates=T
)

--2026-03-13 14:46:20--  https://physionet.org/files/mimic-cxr/2.1.0/SHA256SUMS.txt
Resolving physionet.org (physionet.org)... 18.18.42.54
Connecting to physionet.org (physionet.org)|18.18.42.54|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 76859934 (73M) [text/plain]
Saving to: 'physionet.org/files/mimic-cxr/2.1.0/SHA256SUMS.txt'


physionet.org/files   0%[                    ]       0  --.-KB/s               
physionet.org/files   0%[                    ]  31.49K  78.2KB/s               
physionet.org/files   0%[                    ]  95.49K   158KB/s               
physionet.org/files   0%[                    ] 127.49K   131KB/s               
physionet.org/files   0%[                    ] 191.49K   149KB/s               
physionet.org/files   0%[                    ] 223.49K   146KB/s

### 3.2 `Generate filelist for wget`


In [2]:
from utils.mimic_utils import generate_wget_filelist

# N patients to download
DOWNLOAD_PATIENT_NUM = 5

flist_path = generate_wget_filelist(
    checksum_path = checksum_file,
    patient_num = DOWNLOAD_PATIENT_NUM,
    target_dir = settings.MIMIC_CXR_root,
    templates = T
)



### 3.3 `Download files`

In [ ]:
from utils.mimic_utils import download_wget_filelist

download_wget_filelist(
    username=username,
    password=password,
    filelist = flist_path,
    target_dir = settings.MIMIC_CXR_root,
    templates = T
)



### 3.4 Data Checksum Validation

In [ ]:
from utils.mimic_utils import filelist_checksum_validation

validation_result = filelist_checksum_validation(
    filelist = flist_path,
    target_dir = settings.MIMIC_CXR_root,
    templates = T
)


for error_type in validation_result:
    print(T.MESSAGE_MISSING_TYPE.format(
        error_type = error_type,
        error_num = len(validation_result[error_type])
        ))

files_to_fix = validation_result["wrong checksum"]

Missing files: 0
Mismatched files: 0
OK files: 18876


# 4. `MIMIC-CXR-JPG`

From this dataset we need only diagnostic information in the CSV format - `mimic-cxr-2.0.0-chexpert.csv.gz`.

### 4.1 Download data

In [ ]:
from getpass import getpass
from utils.mimic_utils import download_mimic_iv

username = input("Enter PhysioNet username: ")
password = getpass("Enter PhysioNet password: ")

profile = "default"

settings = DataSetsSettings.build(profile)

download_mimic_iv(
    username=username,
    password=password,
    target_dir=settings.MIMIC_CXR_JPG_root,
    templates=T,
    download_url=T.MIMIC_CXR_JPG_URL
)




### 4.2 Unzip data

In [ ]:
from utils.mimic_utils import unzip_files

unzip_files(
    root_dir=settings.MIMIC_CXR_JPG_root,
    file_extension=T.CSV_GZ_EXTENSION
)



# 5. `Wrong Checksum Correction`

Dowloading files with incorrect checksum

In [7]:
print(files_to_fix)

['files\\p11\\p11255297\\s59219146\\24d13b39-8841b72f-ab094eb1-c7beadbd-73c5b505.dcm', 'files\\p11\\p11306899\\s54410161\\741dcd84-576d21dc-f1880e30-d3cff4aa-e65ebba6.dcm', 'files\\p11\\p11566993\\s52027677\\881a0684-224c15a7-555ea60a-a74466c8-78e6b9db.dcm', 'files\\p11\\p11567818\\s54466415\\1c6708d4-bf59b1f7-975f50d4-781de826-153ef82c.dcm', 'files\\p11\\p11906222\\s57232140\\64927291-fe42a66c-af054049-3d17501b-5de4163c.dcm', 'files\\p11\\p11970980\\s55621312\\21a8e9b8-c3a48789-c7d4e9c4-bd35a79b-f82a6ccf.dcm', 'files\\p12\\p12328230\\s55860441\\0e2b99c5-fbb24588-ed098ccd-e443570a-473ee3b9.dcm', 'files\\p12\\p12513827\\s59251396\\d29628e6-2b5f2fa6-296019f5-25522a07-c62c329f.dcm', 'files\\p12\\p12704088\\s53473937\\824dea05-9d415fa8-e2e766a4-989f3199-10202d9c.dcm', 'files\\p12\\p12889749\\s54479829\\bcd63d38-374a8884-e94cea16-4ab775ed-b92bb4c4.dcm', 'files\\p12\\p12892033\\s57760913\\9a353bd1-51ccbc44-57b72f04-fc7d6c71-ee550ee5.dcm', 'files\\p12\\p12892033\\s57760913\\b0ac07b2-cecf503f-

In [ ]:
from utils.mimic_utils import redownload_wrong_files

status = redownload_wrong_files(
    wrong_files=files_to_fix,
    server=T.MIMIC_CXR_URL,
    location=settings.MIMIC_CXR_root,
    templates=T
)

Downloading: https://physionet.org/files/mimic-cxr/2.1.0/files/p11/p11255297/s59219146/24d13b39-8841b72f-ab094eb1-c7beadbd-73c5b505.dcm
Destination: D:\003.Data\MIMIC-CXR.v2.1-60k\files\p11\p11255297\s59219146\24d13b39-8841b72f-ab094eb1-c7beadbd-73c5b505.dcm
Status: downloaded successfully

Downloading: https://physionet.org/files/mimic-cxr/2.1.0/files/p11/p11306899/s54410161/741dcd84-576d21dc-f1880e30-d3cff4aa-e65ebba6.dcm
Destination: D:\003.Data\MIMIC-CXR.v2.1-60k\files\p11\p11306899\s54410161\741dcd84-576d21dc-f1880e30-d3cff4aa-e65ebba6.dcm
Status: downloaded successfully

Downloading: https://physionet.org/files/mimic-cxr/2.1.0/files/p11/p11566993/s52027677/881a0684-224c15a7-555ea60a-a74466c8-78e6b9db.dcm
Destination: D:\003.Data\MIMIC-CXR.v2.1-60k\files\p11\p11566993\s52027677\881a0684-224c15a7-555ea60a-a74466c8-78e6b9db.dcm
Status: downloaded successfully

Downloading: https://physionet.org/files/mimic-cxr/2.1.0/files/p11/p11567818/s54466415/1c6708d4-bf59b1f7-975f50d4-781de826-1

[{'relative_path': 'files\\p11\\p11255297\\s59219146\\24d13b39-8841b72f-ab094eb1-c7beadbd-73c5b505.dcm',
  'remote_url': 'https://physionet.org/files/mimic-cxr/2.1.0/files/p11/p11255297/s59219146/24d13b39-8841b72f-ab094eb1-c7beadbd-73c5b505.dcm',
  'local_path': 'D:\\003.Data\\MIMIC-CXR.v2.1-60k\\files\\p11\\p11255297\\s59219146\\24d13b39-8841b72f-ab094eb1-c7beadbd-73c5b505.dcm',
  'status': 'downloaded',
  'return_code': 0},
 {'relative_path': 'files\\p11\\p11306899\\s54410161\\741dcd84-576d21dc-f1880e30-d3cff4aa-e65ebba6.dcm',
  'remote_url': 'https://physionet.org/files/mimic-cxr/2.1.0/files/p11/p11306899/s54410161/741dcd84-576d21dc-f1880e30-d3cff4aa-e65ebba6.dcm',
  'local_path': 'D:\\003.Data\\MIMIC-CXR.v2.1-60k\\files\\p11\\p11306899\\s54410161\\741dcd84-576d21dc-f1880e30-d3cff4aa-e65ebba6.dcm',
  'status': 'downloaded',
  'return_code': 0},
 {'relative_path': 'files\\p11\\p11566993\\s52027677\\881a0684-224c15a7-555ea60a-a74466c8-78e6b9db.dcm',
  'remote_url': 'https://physionet.